# Bedrock Agent Tracing Demo

Demonstrates an agentic loop using the **AWS Strands Agents** library (backed by
Amazon Bedrock / Claude 4.5 Sonnet) with two tools — CVE lookup and open-port
checking — while emitting **OpenTelemetry** trajectory traces for every model
call and tool invocation.

Strands manages the agent loop, tool dispatch, and OTel span emission
automatically; the notebook focuses on tool definitions and trajectory display.

**Running modes**

| Mode | What to do |
|------|-----------|
| Fully offline (mocked) | Set `USE_MOCK = True` in Cell 2 and run all cells |
| Live Bedrock | Set `AWS_REGION`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` and keep `USE_MOCK = False` in Cell 2 |
| OTLP export | Set `OTEL_EXPORTER_OTLP_ENDPOINT` before running Cell 3 |


In [ ]:

# ── Cell 2: Configuration ───────────────────────────────────────────────────
import os

# Set USE_MOCK = False and supply AWS credentials to use live Amazon Bedrock.
USE_MOCK   = False
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
MODEL_ID   = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Leave empty to skip OTLP export; set to e.g. "http://localhost:4318" for a collector.
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4318"
OTEL_ENDPOINT = os.environ.get("OTEL_EXPORTER_OTLP_ENDPOINT", "")

print(f"USE_MOCK={USE_MOCK}  MODEL_ID={MODEL_ID}  OTLP={'enabled' if OTEL_ENDPOINT else 'disabled'}")


In [ ]:

# ── Cell 3: OpenTelemetry tracer setup ──────────────────────────────────────
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter, BatchSpanProcessor

_TRACER_PROVIDER: TracerProvider | None = None


def init_tracer(service_name: str = "bedrock-agent-demo") -> trace.Tracer:
    """Idempotent tracer initialisation — safe to call multiple times."""
    global _TRACER_PROVIDER
    if _TRACER_PROVIDER is not None:
        return trace.get_tracer(service_name)

    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

    if OTEL_ENDPOINT:
        from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
        otlp_exporter = OTLPSpanExporter(endpoint=OTEL_ENDPOINT)
        provider.add_span_processor(BatchSpanProcessor(otlp_exporter))
        print(f"OTLP exporter → {OTEL_ENDPOINT}")

    trace.set_tracer_provider(provider)
    _TRACER_PROVIDER = provider
    print(f"Tracer initialised (service={service_name})")
    return trace.get_tracer(service_name)


tracer = init_tracer()


In [ ]:

# ── Cell 4: Mock Strands model ──────────────────────────────────────────────
#
# Deterministic offline stand-in for BedrockModel.
# Emits scripted StreamEvents for three agent turns:
#   1. Model requests lookup_cve
#   2. Model requests check_open_ports
#   3. Model returns a final text answer
#
# The event shapes mirror Bedrock's ConversationStream format, which is what
# Strands' Model.stream() is expected to yield.

from typing import Any, AsyncIterable, Optional
from strands.models import Model
from strands.types.content import Messages
from strands.types.streaming import StreamEvent
from strands.types.tools import ToolSpec


class MockStrandsModel(Model):
    """Deterministic offline stand-in for BedrockModel."""

    _SCRIPT = [
        # Turn 1: request lookup_cve
        [
            {"messageStart": {"role": "assistant"}},
            {"contentBlockStart": {"contentBlockIndex": 0, "start": {
                "toolUse": {"toolUseId": "mock-tool-001", "name": "lookup_cve"}
            }}},
            {"contentBlockDelta": {"contentBlockIndex": 0, "delta": {
                "toolUse": {"input": '{"cve_id": "CVE-2021-44228"}'}
            }}},
            {"contentBlockStop": {"contentBlockIndex": 0}},
            {"messageStop": {"stopReason": "tool_use"}},
            {"metadata": {"usage": {"inputTokens": 120, "outputTokens": 40, "totalTokens": 160},
                          "metrics": {"latencyMs": 10}}},
        ],
        # Turn 2: request check_open_ports
        [
            {"messageStart": {"role": "assistant"}},
            {"contentBlockStart": {"contentBlockIndex": 0, "start": {
                "toolUse": {"toolUseId": "mock-tool-002", "name": "check_open_ports"}
            }}},
            {"contentBlockDelta": {"contentBlockIndex": 0, "delta": {
                "toolUse": {"input": '{"host": "127.0.0.1", "ports": [22, 80, 443, 8080]}'}
            }}},
            {"contentBlockStop": {"contentBlockIndex": 0}},
            {"messageStop": {"stopReason": "tool_use"}},
            {"metadata": {"usage": {"inputTokens": 200, "outputTokens": 55, "totalTokens": 255},
                          "metrics": {"latencyMs": 10}}},
        ],
        # Turn 3: final text answer
        [
            {"messageStart": {"role": "assistant"}},
            {"contentBlockStart": {"contentBlockIndex": 0, "start": {}}},
            {"contentBlockDelta": {"contentBlockIndex": 0, "delta": {"text": (
                "CVE-2021-44228 (Log4Shell) is a critical RCE vulnerability (CVSS 10.0) "
                "in Apache Log4j2. On host 127.0.0.1, the port scan found open/closed "
                "status for ports 22, 80, 443, and 8080. If port 8080 is running a "
                "vulnerable Log4j version, it poses a high risk. "
                "Immediate patching to Log4j 2.17.1+ is strongly recommended."
            )}}},
            {"contentBlockStop": {"contentBlockIndex": 0}},
            {"messageStop": {"stopReason": "end_turn"}},
            {"metadata": {"usage": {"inputTokens": 310, "outputTokens": 120, "totalTokens": 430},
                          "metrics": {"latencyMs": 10}}},
        ],
    ]

    def __init__(self):
        self._call_index = 0

    def reset(self):
        self._call_index = 0

    def update_config(self, **kwargs: Any) -> None:
        pass

    def get_config(self) -> dict:
        return {}

    async def structured_output(self, output_model, prompt, system_prompt=None, **kwargs):
        raise NotImplementedError("MockStrandsModel does not support structured output")

    async def stream(
        self,
        messages: Messages,
        tool_specs: Optional[list[ToolSpec]] = None,
        system_prompt: Optional[str] = None,
        **kwargs: Any,
    ) -> AsyncIterable[StreamEvent]:
        if self._call_index >= len(self._SCRIPT):
            events = [
                {"messageStart": {"role": "assistant"}},
                {"contentBlockStart": {"contentBlockIndex": 0, "start": {}}},
                {"contentBlockDelta": {"contentBlockIndex": 0, "delta": {
                    "text": "I have completed my analysis."
                }}},
                {"contentBlockStop": {"contentBlockIndex": 0}},
                {"messageStop": {"stopReason": "end_turn"}},
                {"metadata": {"usage": {"inputTokens": 50, "outputTokens": 10, "totalTokens": 60},
                              "metrics": {"latencyMs": 10}}},
            ]
        else:
            events = self._SCRIPT[self._call_index]
            self._call_index += 1

        for event in events:
            yield event


_mock_model = MockStrandsModel()
print("Mock Strands model ready.")


In [ ]:

# ── Cell 5: Tool definitions ────────────────────────────────────────────────
import ipaddress
import json
import socket
from strands import tool

# ── Tool 1: CVE lookup ──────────────────────────────────────────────────────

_CVE_DATABASE = {
    "CVE-2021-44228": {
        "id": "CVE-2021-44228",
        "description": "Apache Log4j2 JNDI lookup feature remote code execution (Log4Shell).",
        "cvss_score": 10.0,
        "severity": "CRITICAL",
        "affected": "Apache Log4j2 2.0-beta9 through 2.14.1",
        "fix": "Upgrade to Log4j 2.17.1 or later",
    },
    "CVE-2022-22965": {
        "id": "CVE-2022-22965",
        "description": "Spring Framework RCE via data binding on JDK 9+ (Spring4Shell).",
        "cvss_score": 9.8,
        "severity": "CRITICAL",
        "affected": "Spring Framework < 5.3.18 / 5.2.20",
        "fix": "Upgrade to Spring Framework 5.3.18 or 5.2.20",
    },
    "CVE-2023-44487": {
        "id": "CVE-2023-44487",
        "description": "HTTP/2 Rapid Reset Attack causes denial of service.",
        "cvss_score": 7.5,
        "severity": "HIGH",
        "affected": "Multiple HTTP/2 server implementations",
        "fix": "Apply vendor-specific patches; disable HTTP/2 if not needed",
    },
}


@tool
def lookup_cve(cve_id: str) -> dict:
    """Look up a CVE identifier and return vulnerability details.

    Returns vulnerability information including CVSS score, severity, affected
    versions, and remediation guidance for the given CVE identifier.

    Args:
        cve_id: The CVE identifier, e.g. CVE-2021-44228
    """
    cve_id = cve_id.strip().upper()
    if not cve_id.startswith("CVE-"):
        return {"error": f"Invalid CVE ID format: {cve_id!r}. Expected 'CVE-YYYY-NNNNN'."}
    entry = _CVE_DATABASE.get(cve_id)
    if entry is None:
        return {"cve_id": cve_id, "found": False, "message": "CVE not found in demo database."}
    return {**entry, "found": True}


# ── Tool 2: Open-port checker (loopback only) ───────────────────────────────

@tool
def check_open_ports(host: str, ports: list[int]) -> dict:
    """Check which TCP ports are open on a host.

    Only loopback addresses (127.x.x.x or localhost) are permitted so the demo
    never probes external hosts.

    Args:
        host: IP address or 'localhost' (must be loopback)
        ports: List of TCP port numbers to probe
    """
    if host.lower() == "localhost":
        host = "127.0.0.1"
    try:
        addr = ipaddress.ip_address(host)
    except ValueError:
        return {"error": f"Host {host!r} is not a valid IP address. Only loopback addresses are allowed."}

    if not addr.is_loopback:
        return {"error": f"Host {host} is not a loopback address. Only 127.x.x.x / ::1 are permitted."}

    results: dict[int, str] = {}
    for port in ports:
        if not (1 <= port <= 65535):
            results[port] = "invalid"
            continue
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(0.3)
        try:
            result = sock.connect_ex((str(addr), port))
            results[port] = "open" if result == 0 else "closed"
        except OSError:
            results[port] = "error"
        finally:
            sock.close()
    return {"host": str(addr), "ports": results}


print("Tools registered:", [lookup_cve.tool_name, check_open_ports.tool_name])


In [ ]:

# ── Cell 6: Build Strands agent ─────────────────────────────────────────────
#
# Replaces the manual BedrockClientWrapper.  Strands' Agent handles the model
# call, OTel tracing, and tool dispatch automatically.
# The existing global TracerProvider (Cell 3) is picked up automatically.

from strands import Agent
from strands.models import BedrockModel

_SYSTEM_PROMPT = (
    "You are a security and network diagnostic assistant. "
    "Use the provided tools to investigate CVE vulnerabilities and check local port exposure. "
    "Always invoke both tools when relevant before forming your final answer. "
    "Be concise."
)


def build_agent(use_mock: bool) -> Agent:
    if use_mock:
        _mock_model.reset()
        model = _mock_model
        print("Using MOCK Strands model.")
    else:
        model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
        print(f"Using LIVE BedrockModel (model={MODEL_ID}, region={AWS_REGION}).")
    return Agent(
        model=model,
        tools=[lookup_cve, check_open_ports],
        system_prompt=_SYSTEM_PROMPT,
        # Suppress the default streaming output; Cell 9 prints the summary instead.
        callback_handler=None,
    )


agent = build_agent(USE_MOCK)


In [ ]:

# ── Cell 7: Agent runner ─────────────────────────────────────────────────────
#
# The manual agentic loop is replaced by a single agent(prompt) call.
# Strands handles the model ↔ tool loop internally and emits OTel spans for
# every model invocation and tool execution automatically.
#
# After the run we reconstruct the trajectory summary from the agent's
# message history so the output format stays compatible with Cells 9 and 10.


def run_agent(user_message: str, strands_agent: Agent) -> dict:
    """
    Run the Strands agent for one task and return a trajectory summary.

    Returns a dict with:
      - final_answer (str)
      - tool_calls   (list of {id, name, input, output})
      - turn_count   (int)
    """
    # Clear history so each call starts from a clean state.
    strands_agent.messages = []

    response = strands_agent(user_message)

    # Build a map from toolUseId → raw JSON output (stored in tool-result messages).
    tool_output_map: dict[str, str] = {}
    for msg in strands_agent.messages:
        if msg.get("role") == "user":
            for block in msg.get("content", []):
                if "toolResult" in block:
                    tr = block["toolResult"]
                    for c in tr.get("content", []):
                        if "text" in c:
                            tool_output_map[tr["toolUseId"]] = c["text"]
                            break

    # Collect tool calls from assistant messages (in order).
    tool_calls: list = []
    for msg in strands_agent.messages:
        if msg.get("role") == "assistant":
            for block in msg.get("content", []):
                if "toolUse" in block:
                    tu = block["toolUse"]
                    raw = tool_output_map.get(tu["toolUseId"], "{}")
                    try:
                        output = json.loads(raw)
                    except Exception:
                        output = {"raw": raw}
                    tool_calls.append({
                        "id":    tu["toolUseId"],
                        "name":  tu["name"],
                        "input": tu["input"],
                        "output": output,
                    })

    final_answer = str(response)
    turn_count = sum(1 for msg in strands_agent.messages if msg.get("role") == "assistant")

    return {
        "final_answer": final_answer,
        "tool_calls":   tool_calls,
        "turn_count":   turn_count,
    }


print("Agent runner ready.")


In [ ]:

# ── Cell 8: Run the demo ────────────────────────────────────────────────────
#
# This single prompt exercises BOTH tools in one agent run, so the resulting
# trace hierarchy clearly shows:
#   agent.run  ──►  bedrock.converse  ──►  tool.lookup_cve
#              ──►  bedrock.converse  ──►  tool.check_open_ports
#              ──►  bedrock.converse  (final answer)

DEMO_PROMPT = (
    "I am running a Java web application on 127.0.0.1. "
    "Please look up CVE-2021-44228 and check whether TCP ports 22, 80, 443, "
    "and 8080 are open on that host. Summarise the combined security risk."
)

print("=" * 64)
print("PROMPT:", DEMO_PROMPT)
print("=" * 64)

result = run_agent(DEMO_PROMPT, agent)


In [ ]:

# ── Cell 9: Display trajectory summary & flush spans ───────────────────────

print("\n" + "=" * 64)
print("TRAJECTORY SUMMARY")
print("=" * 64)
print(f"  Turns       : {result['turn_count']}")
print(f"  Tool calls  : {len(result['tool_calls'])}")

for i, tc in enumerate(result["tool_calls"], 1):
    print(f"\n  ── Tool call {i}: {tc['name']} ──")
    print(f"     Input  : {json.dumps(tc['input'])}")
    output_preview = json.dumps(tc["output"])
    print(f"     Output : {output_preview[:200]}{'...' if len(output_preview) > 200 else ''}")

print(f"\n  Final answer:\n  {result['final_answer']}")
print("=" * 64)

# Flush all pending spans to the configured exporters
if _TRACER_PROVIDER is not None:
    _TRACER_PROVIDER.force_flush()
    print("\nAll spans flushed.")


In [ ]:

# ── Cell 10: Inline tests (fully offline) ───────────────────────────────────
#
# All tests use the mock model and local function calls — no AWS calls needed.

import traceback as _tb


def _assert(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)


# ── Tool unit tests ─────────────────────────────────────────────────────────

def test_lookup_cve_known():
    r = lookup_cve("CVE-2021-44228")
    _assert(r["found"] is True, "Expected CVE to be found")
    _assert(r["cvss_score"] == 10.0, f"Expected CVSS 10.0, got {r['cvss_score']}")
    _assert(r["severity"] == "CRITICAL", f"Expected CRITICAL, got {r['severity']}")


def test_lookup_cve_unknown():
    r = lookup_cve("CVE-9999-99999")
    _assert(r["found"] is False, "Expected CVE not found")


def test_lookup_cve_invalid_format():
    r = lookup_cve("NOT-A-CVE")
    _assert("error" in r, "Expected error for invalid format")


def test_check_open_ports_loopback():
    r = check_open_ports("127.0.0.1", [9])  # port 9 (discard) — almost certainly closed
    _assert("host" in r, "Expected 'host' in result")
    _assert("ports" in r, "Expected 'ports' in result")
    _assert(9 in r["ports"], "Expected port 9 in results dict")
    _assert(r["ports"][9] in ("open", "closed"), "Expected open or closed status")


def test_check_open_ports_localhost_alias():
    r = check_open_ports("localhost", [9])
    _assert("host" in r, "Expected 'host' when using 'localhost' alias")


def test_check_open_ports_rejects_external():
    r = check_open_ports("8.8.8.8", [80])
    _assert("error" in r, "Expected error for non-loopback host")


def test_check_open_ports_invalid_ip():
    r = check_open_ports("not-an-ip", [80])
    _assert("error" in r, "Expected error for invalid IP string")


# ── Agent loop tests ────────────────────────────────────────────────────────

def test_mock_agent_run_uses_both_tools():
    mock_agent = build_agent(use_mock=True)
    r = run_agent("Check CVE-2021-44228 and ports on 127.0.0.1.", mock_agent)
    _assert(len(r["tool_calls"]) == 2, f"Expected 2 tool calls, got {len(r['tool_calls'])}")
    tool_names = [tc["name"] for tc in r["tool_calls"]]
    _assert("lookup_cve" in tool_names, "Expected lookup_cve in tool calls")
    _assert("check_open_ports" in tool_names, "Expected check_open_ports in tool calls")


def test_mock_agent_run_returns_final_answer():
    mock_agent = build_agent(use_mock=True)
    r = run_agent("Check CVE-2021-44228.", mock_agent)
    _assert(len(r["final_answer"]) > 0, "Expected non-empty final answer")
    _assert(r["turn_count"] > 0, "Expected at least one assistant turn")


def test_mock_agent_tool_outputs_are_dicts():
    mock_agent = build_agent(use_mock=True)
    r = run_agent("Check CVE-2021-44228.", mock_agent)
    for tc in r["tool_calls"]:
        _assert(isinstance(tc["output"], dict), f"Tool output must be dict, got {type(tc['output'])}")


# ── Runner ──────────────────────────────────────────────────────────────────

_all_tests = [
    test_lookup_cve_known,
    test_lookup_cve_unknown,
    test_lookup_cve_invalid_format,
    test_check_open_ports_loopback,
    test_check_open_ports_localhost_alias,
    test_check_open_ports_rejects_external,
    test_check_open_ports_invalid_ip,
    test_mock_agent_run_uses_both_tools,
    test_mock_agent_run_returns_final_answer,
    test_mock_agent_tool_outputs_are_dicts,
]

_passed = _failed = 0
for _test in _all_tests:
    try:
        _test()
        print(f"  PASS  {_test.__name__}")
        _passed += 1
    except Exception as _exc:
        print(f"  FAIL  {_test.__name__}: {_exc}")
        _tb.print_exc()
        _failed += 1

print(f"\n{'✓' if _failed == 0 else '✗'} {_passed}/{_passed + _failed} tests passed.")


In [ ]:

# ── Cell 11: Reset ──────────────────────────────────────────────────────────
#
# Run this cell to reset demo state without restarting the kernel.
# Then re-run cells 8-9 for a fresh trace.

_mock_model.reset()
agent = build_agent(USE_MOCK)
print("Demo state reset — ready to re-run cells 8–9.")
